In [15]:
#from datetime import datetime
from pathlib import Path
notebook_directory = Path.cwd()

#import numpy as np
import pandas as pd
import xlwings as xw

In [16]:
prices_csv = 'sectors'

start_date = pd.Timestamp("2026-03-25")
end_date   = pd.Timestamp("2026-06-14")

div_month = "202606"

group = 'Sector'

In [17]:
prices_path = (
    notebook_directory.parent.parent
    / "backtesting"
    / "historical prices"
    / f"{prices_csv}.csv"
)
prices_df = pd.read_csv(prices_path, parse_dates=["date"])


db_path = (
    notebook_directory.parent
    / "spreadsheets"
    / "2026 Fin Inst Database.xlsx"
)
wb = xw.Book(db_path)
ws = wb.sheets["Scalar Inputs Table"]
db_df = ws.tables["scalar_inputs_table"].range.options(pd.DataFrame, header=1, index=False).value


anchor_path = (
    notebook_directory.parent
    / "spreadsheets"
    / "2026 Group Trading Inputs.xlsm"
)
wb = xw.Book(anchor_path)
ws = wb.sheets["ANCHOR INPUTs"]
anchor_df = ws.tables["anchor_inputs"].range.options(pd.DataFrame, header=1, index=False).value
anchor_df = anchor_df[anchor_df['group'] == group].copy()


In [18]:
symbols = anchor_df['symbol'].to_list()

In [19]:
#previous_month = (pd.Timestamp.today() - pd.DateOffset(months=1)).to_period("M")
prices_df["date"] = pd.to_datetime(prices_df["date"], errors="coerce")
#prices_df = prices_df.loc[prices_df["date"].dt.to_period("M").eq(previous_month)].copy()
prices_df = prices_df.loc[prices_df["date"].between(start_date, end_date)].copy()

In [20]:
# div_month = (pd.Timestamp.today() - pd.DateOffset(months=1)).strftime("%Y%m")
div_column = f"div {div_month}"

for sym in symbols:
    dividends = db_df.loc[db_df["symbol"].eq(sym), div_column].to_list()
    div_amt = float(dividends[0])

    prices_df[f"{sym}*"] = prices_df[sym] - div_amt

In [21]:
for sym in symbols:
    anchor = anchor_df.loc[anchor_df["symbol"].eq(sym), "anchor"].to_list()[0]
    prices_df[f"{sym}**"] = prices_df[f'{anchor}*'] / prices_df[f'{sym}*']
    avg_ratio = prices_df[f"{sym}**"].mean()
    anchor_df.loc[anchor_df['symbol'] == sym, "multiplier"] = avg_ratio

In [22]:
anchor_df = anchor_df.drop(columns="moving_avg_days")

In [23]:
print(anchor_df)

     group            sector symbol anchor  multiplier
40  Sector  Consumer Staples   FSTA    VDC    4.293903
41  Sector  Consumer Staples    VDC    VDC    1.000000
42  Sector  Consumer Staples    XLP    VDC    2.736230
43  Sector            Energy   FENY    VDE    5.086147
44  Sector            Energy    IYE    VDE    2.670026
45  Sector            Energy    VDE    VDE    1.000000
46  Sector            Energy    XLE    VDE    2.835198
47  Sector        Financials    IYF    VFH    1.023885
48  Sector        Financials    VFH    VFH    1.000000
49  Sector        Financials    XLF    VFH    2.446720
50  Sector        Healthcare   FHLC    VHT    3.870333
51  Sector        Healthcare    IYH    VHT    4.432997
52  Sector        Healthcare    VHT    VHT    1.000000
53  Sector        Healthcare    XLV    VHT    1.867204
54  Sector       Industrials   FIDU    XLI    1.852147
55  Sector       Industrials    XLI    XLI    1.000000
56  Sector       Real Estate   FREL    IYR    3.517323
57  Sector